# Does Qwen3.5-4B represent illegality as something distinct from harmfulness?

A probe is a small linear classifier trained to read a concept off a model's internal state. Several papers show that "safety" probes mostly read harm or topic. This notebook asks whether a probe for illegality reads anything beyond harm.

The data: 60 topics, four sentences each, illegal and harmful, illegal but harmless, legal but harmful, legal and harmless (US law, second person, one sentence). 250 design rows in all (ten cells were refilled after exclusions), of which 6 are excluded, so 244 enter the evaluation. Two check sets are never trained on and are only scored: 60 plain legal acts ("You cook pasta") and 61 negated sentences ("You do not ...", legal by construction). Every one of the 371 rows was read and labelled by hand; the counts are in cell 5.

The test: on the two easy corners (illegal-harmful, legal-harmless) legal means the same as harmless, so a probe trained there cannot tell the two apart. The illegality probe is therefore trained only on harmful sentences and tested only on harmless sentences from topics it never saw; then the reverse; then the same two tests for harm. Layer and regularisation are chosen on 15 validation topics, the 15 test topics are scored once, and the whole procedure is repeated 100 times on labels shuffled within each topic and harm stratum ("beat N of 100").

Two conditions: the model's state is recorded twice, at the last token of the bare sentence, and at the token where the model starts its answer to the question whether the act is illegal (the exact wording is in cell 6). The headline test, illegality trained on harmful and tested on harmless, is run in both conditions. In the run of record the two conditions disagree: the illegality direction stays apart from harm while the model reads the bare sentence and lines up with it at the answer position. That disagreement is the finding. Alongside the headline test: the angle between the illegality and harm directions at every layer against a shuffled band, the illegality direction re-scored with harm projected out, the model's own Yes/No answer as the baseline, word count as a floor, and the two check sets.

Conventions: illegality and harm are the positive ends everywhere. A probe score above zero means "called illegal" (or "called harmful"), and a cosine of +1 between the two directions means they coincide. The JSON files keep legal-positive keys (`cos_dlegal_dharm`, `d_legal`); every printed or plotted number is the illegality-positive version.

Run top to bottom on a GPU runtime. Every code cell has a note above it saying what goes in, what comes out and which line to read. Settings are in cell 2. Cells 1 to 4 are setup and do not count toward the hours; the clocked work starts at cell 5, after the design questions in `journal/design-questions-legality-probe.md` were answered and the prediction written into `journal/highlights.md`. The step-by-step run sheet is the vault note *MATS 12 - Run Sheet (legality probe)*; the column meanings for the hand-check are in `data/SCENARIOS_COLUMNS.md`.

### 1 · GPU check
Runtime → Change runtime type → GPU. A free T4 is enough: the project is one forward pass per sentence and a few minutes of probe fitting.

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip() or "NO GPU. Runtime → Change runtime type → GPU")

### 2 · Settings, the only cell you edit
- `MODEL`: Qwen/Qwen3.5-4B. One model, one run.
- `RUN`: the name for this session's files. The bare condition is saved as `<RUN>`, the prompted condition as `<RUN>_prompted`.
- `VAL_TOPICS`, `TEST_TOPICS`: of the 60 topics, 15 are held out to choose the layer and regularisation and 15 to score once; the remaining 30 train. The split is drawn with seed 0 and was fixed before the run; do not change it after seeing results.
- `DRIVE_DIR`: where results persist on Google Drive.

In [ ]:
MODEL       = "Qwen/Qwen3.5-4B"
RUN         = "lp_4b"
VAL_TOPICS  = 15
TEST_TOPICS = 15
DRIVE_DIR   = "/content/drive/MyDrive/mats12_runs"


### 3 · Get the code and connect Drive (setup, not counted)
Mounts Drive, reads the GitHub token from Colab Secrets (`GITHUB_TOKEN` or `colab-mats12`), clones or pulls the private repo, and runs `scripts/colab_setup.sh`, which installs the packages, links `data/raw` and `data/processed` to Drive and copies `journal` and `figures` back from Drive, so results survive the machine. If the pull fails because of local edits, the cell copies every changed file to a `backup_<time>` folder on Drive, resets the repo, pulls clean, and prints the backup path. If Drive is not mounted (the sign-in popup was blocked), results go to local disk and are lost with the VM; re-run the cell and accept the popup.

In [ ]:
import os, subprocess
from google.colab import drive, userdata
try:
    drive.mount("/content/drive", timeout_ms=90000)   # a Google sign-in popup appears; if it is blocked or ignored, we fall back to local disk
except Exception as e:
    DRIVE_DIR = "/content/mats12_runs"
    print("Drive not mounted (" + str(e)[:80] + "). Using local disk instead: results will NOT survive this machine. Re-run this cell later and accept the popup to persist to Drive.")
os.makedirs(DRIVE_DIR, exist_ok=True)
from getpass import getpass
token = None
for name in ("GITHUB_TOKEN", "colab-mats12"):     # Colab Secrets panel (key icon): either name, "Notebook access" ON
    try:
        token = userdata.get(name)
        if token: break
    except Exception:
        pass
if not token:
    print("No GITHUB_TOKEN found in Colab Secrets. The repo is private, so paste a GitHub fine-grained token")
    print("(GitHub → Settings → Developer settings → Fine-grained tokens; repository: mats12; Contents: read and write).")
    token = getpass("GitHub token: ").strip()
# The token travels in a per-command header, never in the saved remote URL (so it is not written to .git/config).
import base64
AUTH = ["-c", "http.extraheader=AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()]
os.environ["MATS12_GIT_AUTH"] = AUTH[1]
if not os.path.exists("/content/mats12"):
    r = subprocess.run(["git", *AUTH, "clone", "-q", "https://github.com/martinherje/mats12.git", "/content/mats12"], capture_output=True, text=True)
else:
    # The Drive links make git see local changes; autostash carries them across the pull. If that still fails,
    # back up every changed file to Drive, reset, and pull clean; nothing is lost, and the backup path is printed.
    R = "/content/mats12"
    r = subprocess.run(["git", *AUTH, "-C", R, "pull", "--rebase", "--autostash", "-q"], capture_output=True, text=True)
    if r.returncode != 0:
        import shutil, time
        bk = f"{DRIVE_DIR}/backup_{time.strftime('%Y%m%d_%H%M%S')}"
        changed = subprocess.run(["git", "-C", R, "status", "--porcelain"], capture_output=True, text=True).stdout.split("\n")
        for line in changed:
            f = line[3:].strip()
            if f and os.path.isfile(os.path.join(R, f)):
                os.makedirs(os.path.dirname(os.path.join(bk, f)), exist_ok=True); shutil.copy2(os.path.join(R, f), os.path.join(bk, f))
        subprocess.run(["git", "-C", R, "rebase", "--abort"], capture_output=True)
        subprocess.run(["git", "-C", R, "reset", "--hard", "-q"], check=True)
        r = subprocess.run(["git", *AUTH, "-C", R, "pull", "-q"], capture_output=True, text=True)
        print(f"local changes were backed up to {bk} and the repo reset before pulling")
if r.returncode != 0:
    raise SystemExit("git failed: " + r.stderr.replace(token, "<token>").strip() + "\nCheck the token has access to martinherje/mats12 (Contents: read).")
%cd /content/mats12
!git log --oneline -1
!bash scripts/colab_setup.sh "$DRIVE_DIR" 

### 4 · Load the model once (setup, not counted)
Loads the model, runs one forward pass, prints layer count, width and memory. Expect 32 layers and width 2560; the activation files will have 33 layers because the embedding output is stored as layer 0.

In [ ]:
!python scripts/gpu_smoke.py --model $MODEL

### 5 · The dataset
`data/scenarios.csv` holds every sentence the project uses. 371 rows: 250 design rows (60 topics × 4, plus 10 rows that refilled cells emptied by exclusion; per quadrant 62 illegal-harmful, 65 illegal-harmless, 62 legal-harmful, 61 legal-harmless), 60 plain acts and 61 negations. Every row was read by hand with `scripts/tag.py`; among the design rows, 6 are excluded, 6 were relabelled, 66 are flagged borderline on legality, 46 on harm and 23 on both (two negated rows also carry a harm flag; they are never trained on). The 60 legal-harmless rows were replaced on 10 Sep by length-matched legal twins of each topic's illegal-harmless row, because the originals were half the length and word count alone separated them; the originals are kept as the plain-act check set.

This cell runs the validator (counts per quadrant, flags, exclusions, and OK or a list of errors) and prints 12 seeded random design rows with their labels and both borderline flags. Read the sample: if a label looks wrong to you now, fix it with `scripts/tag.py` on your own machine, push, and re-run cell 3. Column meanings: `data/SCENARIOS_COLUMNS.md`. Only `set=main` rows enter the probe; `simple` and `negated` rows are scored once as checks.

In [ ]:
import pandas as pd
!python scripts/validate_scenarios.py data/scenarios.csv
df = pd.read_csv("data/scenarios.csv")
pd.set_option("display.max_colwidth", 120)
print(df[df["set"] == "main"][["id", "quadrant", "topic", "borderline_legal", "borderline_harm", "exclude", "text"]].sample(12, random_state=0).to_string(index=False))
print("\nTo change a label, use scripts/tag.py on your own machine, push, re-run cell 3, then re-run this cell to validate.")

### 6 · Record the model's internal state, in two conditions
One forward pass per row, the residual stream at every layer, saved as `data/processed/acts_<RUN>.npz` and `acts_<RUN>_prompted.npz` with a manifest JSON beside each.

Bare: the sentence on its own; the state at its last token.

Prompted: the sentence inside the question the baseline in cell 9 asks, "Under the law of the United States, is the following action illegal? Answer with exactly one word: Yes or No.", in the chat template with thinking off and the assistant turn opened, so the last token is where the model's answer starts. The prompted probe and the baseline therefore see the same text.

Both conditions are evaluated in full; this notebook does not treat either as the main one. Look for `acts (371, 33, 2560)` twice: 371 rows, 33 layers (embeddings plus 32), width 2560. Open the manifest and confirm model, pooling and template before training anything on the file.

In [ ]:
!python scripts/extract_activations.py --model $MODEL --scenarios data/scenarios.csv --run $RUN --batch-size 16
!python scripts/extract_activations.py --model $MODEL --scenarios data/scenarios.csv --run {RUN}_prompted --batch-size 16 --template chat --generation-prompt --enable-thinking off --question legal


### 7 · The test: train inside one harm stratum, test on the other, on topics never seen (about an hour)
Four designs, each run on both conditions, eight runs in all; on a T4 the run of record took 64 minutes, 6 to 10 minutes per run. For illegality: trained on harmful sentences, tested on harmless ones (`L_h2nh`, the headline test), and the reverse (`L_nh2h`). For harm: trained on illegal sentences, tested on legal ones (`H_i2l`), and the reverse (`H_l2i`). In every run the test topics are ones the probe never saw. Layer and C are chosen on the validation topics only; the test topics are scored once; the interval is a topic-block bootstrap; and the whole selection is repeated 100 times on labels shuffled within each topic × stratum cell.

For each run, the lines to read are `TEST cross-stratum acc … AUROC … beat N/100` and the null line under it. Each run also writes the factorial directions (illegality and harm, from the training topics), the cosine between them at every layer with a label-swap band, the illegality direction scored within each harm stratum on held-out topics with the top 1 to 3 harm components projected out, a word-count-only AUROC on the same test rows, and the two check sets. The `cos(d_illegal, d_harm)` line this cell prints is illegality-positive, the same sign as the tables and the figure.

Everything goes to `data/processed/probeeval_<condition>_<tag>.json`. The cells after this one read these files; the only things computed later are the model's own answers in cell 9, the fair-baseline line in cell 8's table and the by-hand recompute in cell 11.

In [ ]:
DESIGNS = [("legal", "harmful", "L_h2nh"), ("legal", "harmless", "L_nh2h"), ("harmful", "illegal", "H_i2l"), ("harmful", "legal", "H_l2i")]
for cond in [RUN, f"{RUN}_prompted"]:
    for target, stratum, tag in DESIGNS:
        print(f"\n################ {cond} · {tag}: {target} probe trained inside {stratum}, tested on the other stratum ################")
        !python scripts/probe_eval.py --run {cond} --target {target} --train-stratum {stratum} --tag {tag} --val-topics $VAL_TOPICS --test-topics $TEST_TOPICS


### 8 · The results table
`scripts/results_table.py` writes `journal/results.md` from the JSON files: one row per design and condition with the held-out AUROC, how many of the 100 shuffles it beat on AUROC and on accuracy, the accuracy at the fitted cut-off with its interval, and word count alone on the same rows; then a directions table with the layer, cos(d_illegal, d_harm) and its band, the illegality direction within the harmless stratum before and after the top harm component is projected out, and the illegality direction predicting harm; then the check-set counts, the model's own answers from cell 9, and the fair-baseline line, which compares the model's Yes-minus-No logit with the probe on the same 30 held-out sentences. The fair-baseline line is the one thing the script computes; everything else is copied. If cell 9 has not run yet, the just-ask lines say so; re-run this cell after it.

The commented line reruns the headline design with the 3 of 244 rows that contain a legality word dropped. It takes about 8 minutes on a T4 and cannot change the verdict (the result of record is in `results.md`: 241 rows, same AUROC), so it is optional.

In [ ]:
# optional: 3 of 244 rows carry a legality word (s018, s057, s145); about 8 min on a T4; the result of record is in journal/results.md
# !python scripts/probe_eval.py --run $RUN --target legal --train-stratum harmful --tag L_h2nh_nocue --drop-cue-rows --val-topics $VAL_TOPICS --test-topics $TEST_TOPICS
!python scripts/results_table.py --run $RUN

### 9 · Just ask the model
The baseline the probe has to beat or match. Every row (excluded rows dropped, same as the probe) is shown to the model inside the same question the prompted condition used, once for "is this illegal?" and once for "is this harmful?". The one-word answer is scored per quadrant; the Yes-minus-No logit is saved so the model's graded judgement can be scored as a classifier on exactly the held-out rows the probe was tested on, which is the fair-baseline line in cell 8's table. Answers that are not Yes or No are counted separately, not as wrong.

Read the per-quadrant answers, `offdiagonal_accuracy` (the two hard quadrants) and `refused_or_unparsed_frac`. Raw answers go to `data/raw/ask_<RUN>_<label>.jsonl`. The baseline of record for `lp_4b` ran on 10 Sep at 20:45; running this cell again moves the existing raw file to a `.bak` name and produces a second set of answers, so do not re-run it unless you mean to.

In [ ]:
!python scripts/ask_model.py --run $RUN --label legal --model $MODEL
!python scripts/ask_model.py --run $RUN --label harmful --model $MODEL

### 10 · The figure
`scripts/figures.py` draws the one figure, `figures/fig1_<RUN>.png`, from the same JSON files: A, the four tests in both conditions against the shuffled-label 95th percentile; B, cos(d_illegal, d_harm) at every layer, bare and prompted, each with its label-swap band. Inside the band means no more aligned than chance; well above it means illegality and harm share a direction. In the prompted condition the band itself opens to ±0.8 from layer 17, so read the sign and the projection numbers in `results.md`, not the late-layer cosine alone.

In [ ]:
!python scripts/figures.py --run $RUN
from IPython.display import Image, display
display(Image(f"figures/fig1_{RUN}.png", width=900))

### 11 · Recompute the headline by hand, and read the probe's mistakes
Rebuilds the headline number from the saved activations and the saved topic split, line by line: keep the design rows, take the training topics' harmful sentences, fit a standardised logistic regression at the chosen layer and C, score the test topics' harmless sentences. The accuracy and AUROC must equal what cell 7 reported. Then it prints every held-out sentence the probe got wrong with its score (positive means called illegal) and its borderline flags.

Run it twice: once with `COND = RUN` (bare) and once with `COND = f"{RUN}_prompted"`. The prompted run is the half of the reading that has not been recomputed yet: its probe ranks the held-out rows about as well as the bare one but calls every one of them legal, so its accuracy sits at 50% with an interval of 50–50. That is the cut-off failing across strata; the printed rows show it. Read the mistakes: are they the borderline rows, the state-dependent ones, the short ones? Write both recomputed numbers and one paragraph on the mistakes in `journal/verification-log.md`; that paragraph is the answer to the form's "what did you check" question.

The same check runs on the Mac without Colab, since the 4B activations are on the Drive mount, and prints every number next to the script's:

    uv run python notebooks/verify_by_hand.py --run lp_4b --acts-dir "$HOME/Library/CloudStorage/GoogleDrive-mherje@live.com/My Drive/mats12_runs/data/processed"

and the same with `--run lp_4b_prompted`.

In [ ]:
COND = RUN   # or f"{RUN}_prompted": run both and log both in journal/verification-log.md
import numpy as np, pandas as pd, json
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
r = json.load(open(f"data/processed/probeeval_{COND}_L_h2nh.json")); L, C = r["layer"], r["C"]
z = np.load(f"data/processed/acts_{COND}.npz"); acts = z["acts"].astype(np.float32); cols = {k[4:]: z[k] for k in z.files if k.startswith("col_")}
df = pd.DataFrame({k: v for k, v in cols.items()}); df["legal"] = df.legal.astype(int); df["harmful"] = df.harmful.astype(int)
# the design rows only: excluded rows out, and the simple / negated check sets out (they are never trained or tested on)
keep = (df.exclude.astype(int) == 0) & (df["set"].astype(str) == "main"); df, acts = df[keep].reset_index(drop=True), acts[keep.to_numpy()]
train = df.topic.isin(r["split"]["train_topics"]) & (df.harmful == 1)          # illegality probe trained inside the harmful stratum
test  = df.topic.isin(r["split"]["test_topics"])  & (df.harmful == 0)          # tested on the harmless stratum, unseen topics
clf = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=3000)).fit(acts[train.to_numpy(), L], df.legal[train])
pred = clf.predict(acts[test.to_numpy(), L]); truth = df.legal[test].to_numpy()
score = -clf.decision_function(acts[test.to_numpy(), L])                      # + = called illegal
print(f"{COND}: recomputed cross-stratum test accuracy at layer {L}, C={C}: {(pred == truth).mean():.3f}   (script reported {r['test_cross_acc']:.3f}; n = {test.sum()})")
from sklearn.metrics import roc_auc_score
print(f"recomputed AUROC: {roc_auc_score(1 - truth, score):.3f}   (script reported {r['test_cross_auroc']:.3f})")
print(f"called illegal: {int((score > 0).sum())} of {int(test.sum())} held-out harmless-stratum sentences")
wrong = df[test][pred != truth].assign(score=score[pred != truth])
print(f"\n{len(wrong)} mistakes on the {int(test.sum())} held-out harmless-stratum sentences (score + = called illegal; read them, and note which were borderline):")
for _, w in wrong.iterrows(): print(f"  {w.score:+.2f} [{w.quadrant:17s} bl_legal={w.borderline_legal} bl_harm={w.borderline_harm}] {w.text}")

### 12 · Save to Drive and GitHub
Copies `journal/` and `figures/` to Drive, then commits them with the hand-checked `data/scenarios.csv` and pushes. Run it whenever you stop. If the push is rejected as non-fast-forward, the commit is safe on Drive and in the Colab repo; pull on the Mac and push from there.

In [ ]:
# Copies journal and figures to Drive (persistence), then commits them plus the hand-checked dataset to GitHub.
!mkdir -p "$DRIVE_DIR/journal" "$DRIVE_DIR/figures" && cp -r journal/. "$DRIVE_DIR/journal/" && cp -r figures/. "$DRIVE_DIR/figures/"
!git config user.email "mherje@live.com" && git config user.name "Martin Herje"
!git add journal figures data/scenarios.csv && (git commit -qm "journal, figures, scenarios: Colab session" || true) && git -c "$MATS12_GIT_AUTH" push -q origin main && echo pushed